# OsteoLens | Phase 3 | BTXRD classifier training

This standalone Colab notebook resolves the BTXRD file from the public Figshare API, downloads it directly to Colab temporary storage, verifies source metadata and archive integrity, audits actual images/metadata/annotations, creates reproducible grouped splits, and trains/evaluates only after an explicit user gate. Dataset inspection does not require Google Drive; Drive is connected only when you choose to start training and persist outputs.

No trained weights, predictions or metrics are bundled in this notebook.


In [ ]:
# Configuration. ZIP_PATH=None means download directly into /content/BTXRD.zip.
from pathlib import Path

ZIP_PATH = None  # Keep None for the official direct Figshare download.
ARCHIVE_PATH = Path('/content/BTXRD.zip') if ZIP_PATH is None else Path(ZIP_PATH)
EXTRACT_DIR = Path('/content/BTXRD')
LOCAL_OUTPUT_DIR = Path('/content/OsteoLensRuns/BTXRD')
OUTPUT_DIR = LOCAL_OUTPUT_DIR  # audit and split inspection work without Google Drive
FIGSHARE_ARTICLE_ID = 27865398
FIGSHARE_FILE_ID = 50653575
FIGSHARE_API_URL = f'https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}'
EXPECTED_ZIP_BYTES = 840474929
EXPECTED_ZIP_MD5 = '59132a5036d030580ccade8add8f13df'
EXPECTED_SOURCE_ROWS = 3746
SEED = 42
INPUT_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 3e-4
NUM_WORKERS = 2
CONFIRM_TRAINING = False  # Do not change until the audit and split have been reviewed.
print('Archive target:', ARCHIVE_PATH, '| ZIP_PATH:', ZIP_PATH)


## How to use this notebook

Run cells 1–8 in order to download and inspect BTXRD. These steps do not mount or require Google Drive: the archive and extracted files live in Colab temporary `/content` storage, and audit/split files are saved under `/content/OsteoLensRuns/BTXRD`. The downloader first checks the public Figshare article API and confirms the article, file ID, file size, MD5 and official download URL agree with the pinned values below. If the record has changed, it stops instead of silently changing the expected values.

Review the actual class counts, image inventory, annotation audit, and grouped split. Cell 12 mounts Google Drive only when you proceed to training and copy the reviewed audit files there. Keep `CONFIRM_TRAINING = False` until you have reviewed them. Then set it to `True` and run cells 12–15 in order.

Official source: [BTXRD Figshare record](https://doi.org/10.6084/m9.figshare.27865398.v1), article ID 27865398. The dataset paper describes 3,746 images, but the notebook independently counts the downloaded table, unique IDs and actual files; the reported count is a verification target, not proof the archive is complete. [Dataset paper](https://www.nature.com/articles/s41597-024-04311-y).


In [ ]:
# Colab already supplies a CUDA-matched torch/torchvision. Install only auxiliary packages.
%pip install -q 'pandas>=2.2,<4' 'openpyxl>=3.1,<4' 'Pillow>=10.4,<13' 'numpy>=1.26,<3' 'scikit-learn>=1.5,<2' 'matplotlib>=3.9,<4' 'seaborn>=0.13,<1' 'tqdm>=4.66,<5' 'imagehash>=4.3,<5' 'requests>=2.31,<3'

import torch, torchvision
print('Python runtime:', __import__('sys').version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU attached. CPU training is supported but may be very slow; select a GPU runtime if available.')

In [ ]:
# Resolve official Figshare metadata, download with retries/resume, verify, and discover archive structure.
import hashlib, os, re, shutil, time, zipfile
from pathlib import Path
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

class ArchiveDownloadError(RuntimeError): pass
class ArchiveIntegrityError(RuntimeError): pass

# Retry transient API and download failures. Range requests are also handled explicitly below.
http = requests.Session()
retry_policy = Retry(total=5, connect=5, read=5, status=5, backoff_factor=0.8,
    status_forcelist=(429, 500, 502, 503, 504), allowed_methods=frozenset({'GET'}),
    respect_retry_after_header=True, raise_on_status=False)
http.mount('https://', HTTPAdapter(max_retries=retry_policy))
http.mount('http://', HTTPAdapter(max_retries=retry_policy))


def resolve_official_figshare_file():
    """Fetch the public article API and fail if pinned record details do not match."""
    response = http.get(FIGSHARE_API_URL, timeout=(20, 90), allow_redirects=True)
    if not response.ok:
        raise ArchiveDownloadError(f'Figshare API returned HTTP {response.status_code}: {response.url}')
    article = response.json()
    if int(article.get('id', -1)) != FIGSHARE_ARTICLE_ID:
        raise ArchiveIntegrityError(f'Figshare API returned unexpected article ID: {article.get("id")}')
    doi = str(article.get('doi', '')).lower().replace('https://doi.org/', '').rstrip('/')
    if not doi.startswith('10.6084/m9.figshare.27865398'):
        raise ArchiveIntegrityError(f'Figshare DOI does not match the expected BTXRD article: {doi!r}')
    matches = [item for item in article.get('files', []) if int(item.get('id', -1)) == FIGSHARE_FILE_ID]
    if len(matches) != 1:
        raise ArchiveIntegrityError(f'Expected exactly one official file ID {FIGSHARE_FILE_ID}; found {len(matches)}.')
    item = matches[0]
    if not str(item.get('name', '')).lower().endswith('.zip'):
        raise ArchiveIntegrityError(f'Figshare file ID {FIGSHARE_FILE_ID} is not named as a ZIP archive: {item.get("name")!r}')
    api_size = int(item.get('size', -1))
    api_md5 = str(item.get('computed_md5') or item.get('supplied_md5') or '').lower()
    download_url = item.get('download_url')
    if api_size != EXPECTED_ZIP_BYTES:
        raise ArchiveIntegrityError(f'Figshare API size {api_size} differs from pinned {EXPECTED_ZIP_BYTES}; inspect the official record before updating.')
    if not api_md5 or api_md5 != EXPECTED_ZIP_MD5.lower():
        raise ArchiveIntegrityError(f'Figshare API MD5 {api_md5 or "missing"} differs from pinned {EXPECTED_ZIP_MD5}; inspect the official record before updating.')
    if not download_url or not str(download_url).startswith('https://'):
        raise ArchiveIntegrityError('Figshare API did not provide a valid HTTPS download URL for the selected file.')
    print(f'Figshare API verified: article {FIGSHARE_ARTICLE_ID}, file {FIGSHARE_FILE_ID}, '
          f'{api_size:,} bytes, MD5 {api_md5}; URL {download_url}')
    return download_url, item


def file_md5(path):
    digest = hashlib.md5()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def verify_zip_archive(path, expected_size, expected_md5):
    path = Path(path)
    actual_size = path.stat().st_size if path.is_file() else 0
    if actual_size <= 0 or actual_size != expected_size:
        raise ArchiveIntegrityError(f'Archive size {actual_size} differs from expected {expected_size}; empty/incomplete archives are rejected.')
    actual_md5 = file_md5(path)
    if actual_md5.lower() != expected_md5.lower():
        raise ArchiveIntegrityError(f'MD5 mismatch: expected {expected_md5}, received {actual_md5}.')
    if not zipfile.is_zipfile(path):
        raise ArchiveIntegrityError('MD5 passed but the file is not a ZIP archive.')
    try:
        with zipfile.ZipFile(path) as archive:
            bad_member = archive.testzip()  # reads all members and checks their CRCs
    except (OSError, zipfile.BadZipFile, RuntimeError) as exc:
        raise ArchiveIntegrityError(f'ZIP integrity check failed: {exc}') from exc
    if bad_member is not None:
        raise ArchiveIntegrityError(f'ZIP CRC failed for member {bad_member}.')
    return True


def download_verified_archive(url, destination, expected_size, expected_md5, retries=5, show_progress=True):
    """Download to a .part file, resume with Range when supported, promote only after verification."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + '.part')
    if destination.exists():
        try:
            verify_zip_archive(destination, expected_size, expected_md5)
            print(f'Using already verified archive: {destination}')
            return destination
        except ArchiveIntegrityError as exc:
            print(f'Removing invalid existing archive: {exc}')
            destination.unlink(missing_ok=True)
    last_error = None
    for attempt in range(1, retries + 1):
        offset = partial.stat().st_size if partial.exists() else 0
        if offset <= 0 or offset > expected_size:
            partial.unlink(missing_ok=True); offset = 0
        if offset == expected_size:
            try:
                verify_zip_archive(partial, expected_size, expected_md5)
                partial.replace(destination)
                print(f'Download verified: {destination}')
                return destination
            except ArchiveIntegrityError as exc:
                last_error = exc; partial.unlink(missing_ok=True); offset = 0
        headers = {'User-Agent': 'OsteoLens-Colab/1.1', 'Accept-Encoding': 'identity'}
        if offset:
            headers['Range'] = f'bytes={offset}-'
        try:
            with http.get(url, headers=headers, stream=True, timeout=(20, 120), allow_redirects=True) as response:
                if response.status_code not in (200, 206):
                    response.raise_for_status()
                    raise ArchiveDownloadError(f'Unexpected HTTP status {response.status_code} at {response.url}')
                if not response.url.startswith('https://'):
                    raise ArchiveDownloadError(f'Refusing non-HTTPS final download URL: {response.url}')
                if response.status_code == 206:
                    content_range = response.headers.get('Content-Range', '')
                    match = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+|\*)', content_range.strip())
                    if not match or int(match.group(1)) != offset or int(match.group(3)) != expected_size:
                        raise ArchiveDownloadError(f'Invalid Content-Range for resume: {content_range!r}')
                    mode, initial = 'ab', offset
                    expected_response_bytes = expected_size - offset
                else:
                    # A server that ignores Range returns the full body: restart instead of appending.
                    mode, initial = 'wb', 0
                    expected_response_bytes = expected_size
                content_length = response.headers.get('Content-Length')
                if content_length is not None and int(content_length) != expected_response_bytes:
                    raise ArchiveDownloadError(f'HTTP Content-Length {content_length} differs from expected response bytes {expected_response_bytes}.')
                content_type = response.headers.get('Content-Type', '').lower()
                if 'text/html' in content_type:
                    raise ArchiveDownloadError(f'Expected archive bytes, received HTML from {response.url}')
                with tqdm(total=expected_size, initial=initial, unit='B', unit_scale=True,
                          desc=f'BTXRD ({attempt}/{retries})', disable=not show_progress) as bar:
                    received_this_response = 0
                    with partial.open(mode) as output:
                        for block in response.iter_content(chunk_size=1024 * 1024):
                            if not block: continue
                            output.write(block); received_this_response += len(block); bar.update(len(block))
                            if output.tell() > expected_size:
                                raise ArchiveDownloadError('Download exceeded the API-verified archive size.')
                    output_size = partial.stat().st_size
                if received_this_response != expected_response_bytes or output_size != expected_size:
                    raise ArchiveDownloadError(f'Incomplete HTTP body: received {received_this_response} response bytes; archive has {output_size}/{expected_size}.')
            verify_zip_archive(partial, expected_size, expected_md5)
            partial.replace(destination)
            print(f'Download, HTTP, size, API MD5, ZIP and CRC checks passed: {destination}')
            return destination
        except Exception as exc:
            last_error = exc
            if isinstance(exc, ArchiveIntegrityError):
                partial.unlink(missing_ok=True)  # never resume a complete but corrupt copy
            if attempt < retries:
                delay = min(2 ** (attempt - 1), 16)
                print(f'Attempt {attempt}/{retries} failed: {exc}. Retry in {delay}s.')
                time.sleep(delay)
    # Failed calls leave no empty/stale archive. A later notebook run starts cleanly.
    partial.unlink(missing_ok=True)
    destination.unlink(missing_ok=True)
    raise ArchiveDownloadError(f'BTXRD download failed after {retries} attempts; incomplete files removed. Last error: {last_error}') from last_error


def available_memory_bytes():
    try:
        for line in Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'): return int(line.split()[1]) * 1024
    except OSError:
        return None
    return None


def check_resources(stage, minimum_disk_bytes=0, minimum_ram_bytes=0, root='/content'):
    disk_free = shutil.disk_usage(root).free
    ram_free = available_memory_bytes()
    ram_text = f'{ram_free / 2**30:.2f} GiB' if ram_free is not None else 'unavailable'
    print(f'{stage}: free disk {disk_free / 2**30:.2f} GiB; available RAM {ram_text}')
    if disk_free < minimum_disk_bytes:
        raise RuntimeError(f'Insufficient Colab disk for {stage}: need {minimum_disk_bytes / 2**30:.2f} GiB free.')
    if ram_free is not None and ram_free < minimum_ram_bytes:
        raise RuntimeError(f'Insufficient RAM for {stage}: need {minimum_ram_bytes / 2**30:.2f} GiB available.')
    return disk_free, ram_free


def safe_extract_zip(zip_path, target_dir):
    target_dir = Path(target_dir); target_dir.mkdir(parents=True, exist_ok=True)
    root = target_dir.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            destination = (target_dir / member.filename).resolve()
            if root not in destination.parents and destination != root:
                raise ArchiveIntegrityError(f'Unsafe path in ZIP: {member.filename}')
        archive.extractall(target_dir)


def find_named_dir(root, wanted):
    for current, dirs, _ in os.walk(root):
        dirs.sort()
        for dirname in dirs:
            if dirname.lower() == wanted.lower(): return Path(current) / dirname
    return None


def find_metadata_file(root):
    accepted = {'dataset.csv', 'dataset.xlsx', 'dataset.xls'}
    for current, _, files in os.walk(root):
        for name in sorted(files, key=str.lower):
            if name.lower() in accepted: return Path(current) / name
    return None

# Authenticate the values from Figshare's public API before requesting any archive bytes.
FIGSHARE_URL, FIGSHARE_FILE_METADATA = resolve_official_figshare_file()
if ZIP_PATH is None:
    print(f'Downloading official BTXRD archive directly to {ARCHIVE_PATH} (Colab temporary storage)...')
else:
    print(f'Using explicitly supplied ZIP_PATH: {ARCHIVE_PATH}')
check_resources('download', 2 * 2**30 if ARCHIVE_PATH.exists() else EXPECTED_ZIP_BYTES + 2 * 2**30, 2 * 2**30)
archive_path = download_verified_archive(FIGSHARE_URL, ARCHIVE_PATH, EXPECTED_ZIP_BYTES, EXPECTED_ZIP_MD5)

# Reuse a complete extraction only when it was made from this exact verified archive.
ready_marker = EXTRACT_DIR / '.osteolens_btxrd_ready'
if not (EXTRACT_DIR.exists() and ready_marker.exists() and ready_marker.read_text().strip() == EXPECTED_ZIP_MD5):
    if EXTRACT_DIR.exists(): shutil.rmtree(EXTRACT_DIR)
    staging = Path('/content/BTXRD.extracting')
    if staging.exists(): shutil.rmtree(staging)
    with zipfile.ZipFile(archive_path) as archive:
        expanded_bytes = sum(item.file_size for item in archive.infolist())
    check_resources('extraction', expanded_bytes + 2 * 2**30, 2 * 2**30)
    print(f'Extracting {expanded_bytes / 2**30:.2f} GiB to Colab temporary storage...')
    safe_extract_zip(archive_path, staging)
    if find_named_dir(staging, 'images') is None or find_metadata_file(staging) is None:
        shutil.rmtree(staging, ignore_errors=True)
        raise FileNotFoundError('Archive lacks a recognizable images folder or dataset.csv/xlsx.')
    staging.replace(EXTRACT_DIR)
    (EXTRACT_DIR / '.osteolens_btxrd_ready').write_text(EXPECTED_ZIP_MD5)

search_root = EXTRACT_DIR
images_dir = find_named_dir(search_root, 'images')
annotations_dir = find_named_dir(search_root, 'annotations')
metadata_path = find_metadata_file(search_root)
if images_dir is None or metadata_path is None:
    raise FileNotFoundError('Verified extraction is missing images/ or a dataset.csv/xlsx metadata file.')
print('Dataset root:', search_root)
print('Image folder:', images_dir)
print('Metadata file:', metadata_path)
print('Annotations:', annotations_dir if annotations_dir else 'not present')


In [ ]:
# Load the archive metadata and map its source flags to the three broad source labels.
import re, numpy as np, pandas as pd

metadata = pd.read_csv(metadata_path) if metadata_path.suffix.lower() == '.csv' else pd.read_excel(metadata_path)
metadata.columns = [str(c).strip() for c in metadata.columns]
print('Metadata rows/columns:', metadata.shape)
print('Columns:', metadata.columns.tolist())
display(metadata.head(8))

CLASS_NAMES = ('Normal', 'Benign', 'Malignant')
def norm_col(value): return re.sub(r'[^a-z0-9]+', '_', str(value).strip().lower()).strip('_')
column_map = {norm_col(c): c for c in metadata.columns}
def flag_state(value):
    if pd.isna(value): return None
    if isinstance(value, (int, float, np.number)):
        return True if value == 1 else False if value == 0 else None
    raw = str(value).strip().lower()
    if raw in {'1','true','yes','y'}: return True
    if raw in {'0','false','no','n'}: return False
    return None
def source_class(row):
    for key in ('classification','class','broad_class','diagnosis'):
        if key in column_map and pd.notna(row[column_map[key]]):
            raw = str(row[column_map[key]]).strip().lower()
            if raw in {'normal','normal bone'}: return 'Normal'
            if 'benign' in raw: return 'Benign'
            if 'malignant' in raw: return 'Malignant'
    b = flag_state(row[column_map['benign']]) if 'benign' in column_map else None
    m = flag_state(row[column_map['malignant']]) if 'malignant' in column_map else None
    t = flag_state(row[column_map['tumor']]) if 'tumor' in column_map else None
    if b is True and m is True: return None
    if m is True: return 'Malignant'
    if b is True: return 'Benign'
    if t is False: return 'Normal'
    return None

image_id_col = next((column_map[k] for k in ('image_id','image','filename') if k in column_map), None)
if image_id_col is None: raise ValueError('No image_id/image/filename column found. Inspect metadata schema before proceeding.')
metadata['image_id'] = metadata[image_id_col].astype(str).str.strip()
metadata['label'] = metadata.apply(source_class, axis=1)
def image_stem(value):
    if pd.isna(value) or not str(value).strip(): return None
    return Path(str(value).replace('\\','/')).stem.casefold()
metadata['image_key'] = metadata.image_id.map(image_stem)

# Index actual files recursively, matching metadata by filename stem (case-insensitive).
from collections import defaultdict
image_files = sorted(p for p in images_dir.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg'})
file_index = defaultdict(list)
for image_file in image_files: file_index[image_file.stem.casefold()].append(image_file)
def resolve_image(key):
    matches = file_index.get(key, []) if key else []
    return matches[0] if len(matches) == 1 else None
metadata['image_path'] = metadata.image_key.map(resolve_image)
metadata['image_exists'] = metadata.image_path.map(lambda p: p is not None and p.is_file())
metadata_dupe_rows = int(metadata.image_key.duplicated(keep=False).sum())
duplicate_file_stems = {key: paths for key, paths in file_index.items() if len(paths) > 1}
metadata_keys = set(metadata.image_key.dropna())
image_keys = set(file_index)
missing_image_ids = metadata.loc[~metadata.image_exists, 'image_id'].tolist()
unreferenced_image_files = [str(p) for k, paths in file_index.items() if k not in metadata_keys for p in paths]
ambiguous_image_keys = sorted(set(file_index) & {k for k, paths in file_index.items() if len(paths) > 1})

print('Metadata rows:', len(metadata), '| unique nonempty IDs:', metadata.image_key.nunique(dropna=True))
print('Physical JPEG/JPG files:', len(image_files), '| duplicate stems:', len(duplicate_file_stems))
print('Missing metadata-referenced files:', len(missing_image_ids), '| unreferenced files:', len(unreferenced_image_files))
print('Source labels (computed from this table):')
display(metadata.label.value_counts(dropna=False).rename_axis('source label').to_frame('records'))
if missing_image_ids: display(pd.DataFrame({'missing_image_id': missing_image_ids[:30]}))
if unreferenced_image_files: display(pd.DataFrame({'unreferenced_file': unreferenced_image_files[:30]}))

In [ ]:
# Read every JPEG/JPG, record actual format/mode/dimensions, and audit annotations against tumor-labeled records.
from PIL import Image, ImageOps, UnidentifiedImageError
from collections import Counter
import json

image_audit_rows = []
for image_file in image_files:
    try:
        with Image.open(image_file) as im: im.verify()
        with Image.open(image_file) as im:
            image_audit_rows.append({'file': str(image_file), 'image_id': image_file.name,
                'stem': image_file.stem.casefold(), 'format': im.format, 'mode': im.mode,
                'width': im.width, 'height': im.height, 'readable': True, 'error': None})
    except (OSError, UnidentifiedImageError) as exc:
        image_audit_rows.append({'file': str(image_file), 'image_id': image_file.name,
            'stem': image_file.stem.casefold(), 'format': None, 'mode': None,
            'width': None, 'height': None, 'readable': False, 'error': str(exc)})
image_audit = pd.DataFrame(image_audit_rows)
unreadable_image_ids = image_audit.loc[~image_audit.readable, 'image_id'].tolist()
print('Readable images:', int(image_audit.readable.sum()), '/', len(image_audit))
print('Image formats:', image_audit['format'].value_counts(dropna=False).to_dict())
print('Image modes:', image_audit['mode'].value_counts(dropna=False).to_dict())
if len(image_audit): display(image_audit.groupby('format')[['width','height']].agg(['min','median','max']))
if unreadable_image_ids: display(pd.DataFrame({'unreadable_image_id': unreadable_image_ids[:30]}))

annotation_files = sorted(p for p in annotations_dir.rglob('*') if p.is_file() and p.suffix.lower()=='.json') if annotations_dir else []
annotation_stems = [p.stem.casefold() for p in annotation_files]
annotation_summary = Counter()
annotation_examples = []
valid_annotation_stems = set()
for json_path in annotation_files:
    try:
        document = json.loads(json_path.read_text(encoding='utf-8'))
        if not isinstance(document, dict) or not isinstance(document.get('shapes', []), list):
            raise ValueError('Expected a LabelMe object with a shapes list')
        valid_annotation_stems.add(json_path.stem.casefold())
        shapes = document.get('shapes', [])
        annotation_summary['valid_json_files'] += 1
        annotation_summary['shapes_total'] += len(shapes)
        annotation_summary['embedded_image_data'] += bool(document.get('imageData'))
        annotation_summary['image_dimensions_present'] += bool(document.get('imageWidth') and document.get('imageHeight'))
        for shape in shapes:
            annotation_summary[f"shape_type:{shape.get('shape_type','missing')}"] += 1
            annotation_summary[f"label:{shape.get('label','missing')}"] += 1
            if not shape.get('points'): annotation_summary['shapes_without_points'] += 1
        if len(annotation_examples) < 8:
            annotation_examples.append({'file':json_path.name,'imagePath':document.get('imagePath'),
                'imageWidth':document.get('imageWidth'),'imageHeight':document.get('imageHeight'),
                'shapes':[{'label':s.get('label'),'shape_type':s.get('shape_type'),
                           'point_count':len(s.get('points',[]))} for s in shapes[:8]]})
    except (OSError, json.JSONDecodeError, ValueError, TypeError) as exc:
        annotation_summary['invalid_json_files'] += 1
        print('Invalid annotation:', json_path, '|', exc)
metadata_labels_by_key = metadata.dropna(subset=['image_key']).drop_duplicates('image_key').set_index('image_key').label.to_dict()
tumor_keys = {key for key, label in metadata_labels_by_key.items() if label in {'Benign','Malignant'}}
annotation_keys = set(annotation_stems)
matched_tumor_annotations = tumor_keys & valid_annotation_stems
missing_tumor_annotations = sorted(tumor_keys - valid_annotation_stems)
orphan_annotation_files = sorted(annotation_keys - metadata_keys)
annotation_summary.update({
    'annotation_json_files_found': len(annotation_files),
    'unique_annotation_stems': len(annotation_keys),
    'duplicate_annotation_stems': len(annotation_stems) - len(annotation_keys),
    'tumor_labeled_records_from_metadata': len(tumor_keys),
    'matched_tumor_annotations': len(matched_tumor_annotations),
    'tumor_records_missing_annotation': len(missing_tumor_annotations),
    'annotation_files_without_metadata_record': len(orphan_annotation_files),
})
print('Annotation audit (counts read from this archive):', dict(annotation_summary))
display(pd.DataFrame(annotation_examples))
if missing_tumor_annotations: display(pd.DataFrame({'tumor_record_missing_annotation': missing_tumor_annotations[:30]}))

# Persist the source audit and complete per-image inventory to Drive, not the data archive.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
image_audit.to_csv(OUTPUT_DIR / 'image_audit.csv', index=False)
audit = {
    'source': 'BTXRD Figshare file 50653575', 'archive_md5_verified': EXPECTED_ZIP_MD5,
    'metadata_file': metadata_path.name, 'metadata_rows': int(len(metadata)),
    'metadata_unique_image_ids': int(metadata.image_key.nunique(dropna=True)),
    'physical_jpeg_files': int(len(image_files)), 'missing_image_count': int(len(missing_image_ids)),
    'unreferenced_image_count': int(len(unreferenced_image_files)),
    'duplicate_metadata_id_rows': metadata_dupe_rows, 'duplicate_image_stems': int(len(duplicate_file_stems)),
    'unreadable_image_count': int(len(unreadable_image_ids)),
    'class_counts': {str(k): int(v) for k,v in metadata.label.value_counts(dropna=False).items()},
    'annotation_statistics': dict(annotation_summary),
    'missing_image_ids': missing_image_ids, 'unreadable_image_ids': unreadable_image_ids,
    'unreferenced_image_files': unreferenced_image_files,
    'missing_tumor_annotations': missing_tumor_annotations,
    'orphan_annotation_files': orphan_annotation_files,
}
(OUTPUT_DIR / 'dataset_audit.json').write_text(json.dumps(audit, indent=2, default=str), encoding='utf-8')
print('Saved dataset audit and image inventory to Drive:', OUTPUT_DIR)

## Inspect completeness before creating splits

The paper's 3,746 count is only a reference. The next cell independently requires 3,746 table rows, 3,746 unique source IDs, 3,746 physical JPEG/JPG files, one-to-one matches, readable images, and a valid source class for every row. Class counts are calculated from the downloaded table and are not filled in from reported totals. Annotation coverage is audited separately: missing tumor annotations do not get invented and are explicitly reported. Any incomplete image/label release stops the training path.

In [ ]:
# Stop here unless this exact archive has a complete one-to-one image/metadata audit.
actual_counts = {
    'metadata_rows': len(metadata),
    'unique_metadata_ids': metadata.image_key.nunique(dropna=True),
    'image_files': len(image_files),
    'matched_metadata_images': int(metadata.image_exists.sum()),
    'missing_images': len(missing_image_ids),
    'unreferenced_images': len(unreferenced_image_files),
    'duplicate_metadata_id_rows': metadata_dupe_rows,
    'duplicate_image_stems': len(duplicate_file_stems),
    'unreadable_images': len(unreadable_image_ids),
    'missing_or_ambiguous_labels': int(metadata.label.isna().sum()),
}
print(json.dumps(actual_counts, indent=2))
if any(actual_counts[key] != EXPECTED_SOURCE_ROWS for key in ('metadata_rows','unique_metadata_ids','image_files','matched_metadata_images')):
    raise RuntimeError(f'BTXRD completeness check failed: expected {EXPECTED_SOURCE_ROWS} for each row/ID/image count. See dataset_audit.json.')
if any(actual_counts[key] != 0 for key in ('missing_images','unreferenced_images','duplicate_metadata_id_rows','duplicate_image_stems','unreadable_images','missing_or_ambiguous_labels')):
    raise RuntimeError('BTXRD image or label audit has unresolved records. Inspect Drive audit files; no training was started.')

# Attach readability using actual image filename, then make fixed-seed grouped splits.
readable_names = set(image_audit.loc[image_audit.readable, 'image_id'].str.casefold())
usable = metadata.copy()
usable = usable[usable.image_path.map(lambda p: p.name.casefold() in readable_names)].reset_index(drop=True)
if set(usable.label.unique()) != set(CLASS_NAMES):
    raise RuntimeError(f'Expected all three source classes; found {usable.label.value_counts().to_dict()}')

from sklearn.model_selection import StratifiedGroupKFold
import imagehash
from PIL import Image

# Hash every image and combine patient/study groups with exact and near-duplicate groups.
normalized_columns = {norm_col(c): c for c in usable.columns}
group_col = next((normalized_columns[k] for k in ('patient_id','subject_id','study_id','patient','subject','study')
    if k in normalized_columns and usable[normalized_columns[k]].notna().any()
    and usable[normalized_columns[k]].nunique(dropna=True) < len(usable)), None)
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''): digest.update(block)
    return digest.hexdigest()
usable['sha256'] = usable.image_path.map(sha256_file)
phashes = []
for path in usable.image_path:
    with Image.open(path) as im:
        phashes.append(int(str(imagehash.phash(ImageOps.exif_transpose(im).convert('L'))),16))
parent = list(range(len(usable)))
def root_of(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
def union(i,j):
    ri,rj = root_of(i),root_of(j)
    if ri != rj: parent[rj] = ri
# First keep every record from the same source patient/subject/study together.
if group_col:
    patient_keys = usable[group_col].astype('string').fillna('unlinked-' + usable.image_key)
    for _, indices in usable.groupby(patient_keys, sort=False).groups.items():
        indices = list(indices)
        for index in indices[1:]: union(indices[0], index)
# Then join exact/near duplicates across all source groups as an additional leakage guard.
duplicate_label_conflicts = []
for i, value in enumerate(phashes):
    for j in range(i):
        if (value ^ phashes[j]).bit_count() <= 4:
            if usable.iloc[i].label != usable.iloc[j].label:
                duplicate_label_conflicts.append((usable.iloc[i].image_id,usable.iloc[j].image_id))
            union(i,j)
if duplicate_label_conflicts:
    raise RuntimeError(f'Near-duplicate images have conflicting source labels: {duplicate_label_conflicts[:10]}. Review before training.')
usable['split_group'] = [f'group-{root_of(i)}' for i in range(len(usable))]
patient_grouping_verified = bool(group_col and norm_col(group_col) in {'patient_id','subject_id','patient','subject'} and usable[group_col].notna().all())
split_basis = (f'{group_col} + SHA-256/pHash duplicate clusters' if group_col
               else 'SHA-256/pHash duplicate clusters; patient IDs unavailable')
if not patient_grouping_verified:
    print('LIMITATION: no verified patient/subject key; duplicate grouping cannot guarantee different views from one patient remain together.')
print('Split grouping:',split_basis,'| source group field:',group_col or 'unavailable')
print('Source groups containing multiple classes:',int(usable.assign(_src_group=(patient_keys if group_col else usable.image_key)).groupby('_src_group').label.nunique().gt(1).sum()))

y = usable.label.map({name:i for i,name in enumerate(CLASS_NAMES)}).to_numpy()
groups = usable.split_group.to_numpy()
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trainval_i, test_i = next(outer.split(usable,y,groups))
trainval = usable.iloc[trainval_i].reset_index(drop=True)
y_tv = trainval.label.map({name:i for i,name in enumerate(CLASS_NAMES)}).to_numpy()
g_tv = trainval.split_group.to_numpy()
inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED+1)
train_i, val_i = next(inner.split(trainval,y_tv,g_tv))
train_df, val_df, test_df = trainval.iloc[train_i].copy(), trainval.iloc[val_i].copy(), usable.iloc[test_i].copy()
splits = pd.concat([train_df.assign(split='train'),val_df.assign(split='validation'),test_df.assign(split='test')])
assert not (set(train_df.split_group)&set(val_df.split_group))
assert not (set(train_df.split_group)&set(test_df.split_group))
assert not (set(val_df.split_group)&set(test_df.split_group))
for split_name, frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    if set(frame.label.unique()) != set(CLASS_NAMES):
        raise RuntimeError(f'{split_name} split does not contain all three classes; adjust grouping/stratification before training.')
print('Split sizes by source label:'); display(pd.crosstab(splits.split,splits.label))
splits[['image_id','label','split_group','split','sha256']].to_csv(OUTPUT_DIR/'split_manifest.csv',index=False)
print('Split manifest saved to Drive:', OUTPUT_DIR/'split_manifest.csv')

## Preprocessing and split review

Preprocessing converts the original image to RGB, preserves aspect ratio by padding to a square, resizes to 256 × 256, then applies the fixed ImageNet normalization. No aggressive flips or synthetic medical findings are applied. CrossEntropy loss uses class weights computed only from the training split.

A verified patient/subject key is used when present. If not, exact SHA-256 values and perceptual-hash near-duplicates are kept in one split. This reduces image duplication leakage but **cannot** establish patient-independent evaluation if the dataset has no patient identifier. The notebook records that limitation in the split manifest and metrics.

In [ ]:
# Create preprocessing pipeline and inspect actual preprocessed training images.
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

NORMALIZE_MEAN = (0.485,0.456,0.406)
NORMALIZE_STD = (0.229,0.224,0.225)
PREPROCESSING_CONFIG = {
    'color_conversion':'RGB', 'resize_method':'aspect_preserving_pad', 'input_size':INPUT_SIZE,
    'interpolation':'Pillow.Resampling.LANCZOS', 'padding_rgb':[0,0,0],
    'normalization_mean':list(NORMALIZE_MEAN), 'normalization_std':list(NORMALIZE_STD),
}
tensor_transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize(NORMALIZE_MEAN,NORMALIZE_STD)])
class BTXRDDataset(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as source:
            image = ImageOps.exif_transpose(source).convert('RGB')
            image = ImageOps.pad(image,(INPUT_SIZE,INPUT_SIZE),method=Image.Resampling.LANCZOS,color=(0,0,0))
        return tensor_transform(image), CLASS_NAMES.index(row.label), row.image_id
train_ds,val_ds,test_ds = BTXRDDataset(train_df),BTXRDDataset(val_df),BTXRDDataset(test_df)
train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
fig,axes = plt.subplots(1,min(4,len(train_ds)),figsize=(12,3))
for ax,i in zip(np.atleast_1d(axes),range(min(4,len(train_ds)))):
    tensor,label_id,image_id = train_ds[i]
    shown = tensor.permute(1,2,0).numpy()*np.array(NORMALIZE_STD)+np.array(NORMALIZE_MEAN)
    ax.imshow(np.clip(shown,0,1)); ax.set_title(f'{image_id}\n{CLASS_NAMES[label_id]}'); ax.axis('off')
plt.tight_layout(); plt.show()

## Confirm split and runtime before training

The next cell checks available RAM, free disk, and GPU memory. If GPU memory is tight, it reduces batch size; if the runtime cannot support a safe run, it stops with the resource amounts. Colab GPU availability and session duration vary by account. A T4 or comparable GPU is recommended; CPU training may take a long time.

In [ ]:
# Google Drive is needed only now, after local Colab audit/split review, to preserve training outputs.
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/OsteoLensRuns/BTXRD')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for audit_name in ('dataset_audit.json', 'image_audit.csv', 'split_manifest.csv'):
    source = LOCAL_OUTPUT_DIR / audit_name
    if source.exists(): shutil.copy2(source, OUTPUT_DIR / audit_name)
print('Reviewed audit and split copied to Drive:', OUTPUT_DIR)

# Runtime resource check and model initialization. Colab torch/torchvision are not replaced.
check_resources('training',minimum_disk_bytes=1*2**30,minimum_ram_bytes=3*2**30)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    free_gpu,total_gpu = torch.cuda.mem_get_info()
    print(f'GPU memory: {free_gpu/2**30:.2f} GiB free of {total_gpu/2**30:.2f} GiB')
    if free_gpu < 2*2**30: raise RuntimeError('Less than 2 GiB GPU memory is free. Restart runtime or select a larger GPU.')
    if free_gpu < 4*2**30: BATCH_SIZE = min(BATCH_SIZE,8)
    elif free_gpu < 7*2**30: BATCH_SIZE = min(BATCH_SIZE,16)
else:
    print('CPU mode selected; training can be slow.')
print('Device:',device,'| batch size:',BATCH_SIZE)

# ImageNet weights initialize the network only; the classification head is trained on BTXRD.
weights = torchvision.models.ResNet18_Weights.DEFAULT
model = torchvision.models.resnet18(weights=weights)
model.fc = torch.nn.Linear(model.fc.in_features,len(CLASS_NAMES))
model = model.to(device)
train_counts = train_df.label.value_counts().reindex(CLASS_NAMES,fill_value=0).to_numpy(dtype=np.float32)
if np.any(train_counts==0): raise RuntimeError(f'Train split is missing a class: {train_counts}')
class_weights = torch.tensor(len(train_df)/(len(CLASS_NAMES)*train_counts),dtype=torch.float32,device=device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=0.5,patience=2)
scaler = torch.amp.GradScaler('cuda',enabled=(device.type=='cuda'))
print('Training class counts:',dict(zip(CLASS_NAMES,train_counts.astype(int))))
print('Class weights:',dict(zip(CLASS_NAMES,class_weights.detach().cpu().tolist())))

In [ ]:
# TRAINING CELL — run only after reviewing the audit and split, then setting CONFIRM_TRAINING = True.
if not CONFIRM_TRAINING:
    raise RuntimeError('Training is gated. Review dataset_audit.json and split_manifest.csv, set CONFIRM_TRAINING=True in the setup cell, then run this cell.')

import copy, random
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

MODEL_VERSION = f'btxrd-resnet18-seed{SEED}'
checkpoint_path = OUTPUT_DIR/'osteolens_btxrd_resnet18.pth'
def checkpoint_payload(state_dict,best_epoch,best_val_f1):
    return {'architecture':'resnet18','state_dict':state_dict,'class_names':list(CLASS_NAMES),
        'input_size':INPUT_SIZE,'model_version':MODEL_VERSION,'best_epoch':int(best_epoch),
        'best_validation_macro_f1':float(best_val_f1),'seed':int(SEED),'dataset':'BTXRD',
        'dataset_doi':'10.6084/m9.figshare.27865398.v1','dataset_md5':EXPECTED_ZIP_MD5,
        'dataset_license':'CC BY 4.0; retain attribution',
        'split_grouping':split_basis,'patient_independent_split_verified':bool(patient_grouping_verified),
        'batch_size':int(BATCH_SIZE),'epochs_requested':int(EPOCHS),'learning_rate':float(LEARNING_RATE),
        'runtime_device':str(device),'torch_version':torch.__version__,'torchvision_version':torchvision.__version__,
        'preprocessing_config':PREPROCESSING_CONFIG}
def atomic_save_checkpoint(payload,path):
    temp=path.with_suffix(path.suffix+'.tmp')
    torch.save(payload,temp); os.replace(temp,path)

history=[]; best_val_f1=-1.0; best_epoch=0; best_state=None
for epoch in range(1,EPOCHS+1):
    model.train(); total_loss=0.0; ytr=[]; ptr=[]
    for images,labels,_ in tqdm(train_loader,desc=f'Epoch {epoch}/{EPOCHS}',leave=False):
        images,labels=images.to(device,non_blocking=True),labels.to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type,enabled=(device.type=='cuda')):
            logits=model(images); loss=criterion(logits,labels)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total_loss+=loss.item()*labels.size(0); ytr.extend(labels.cpu().tolist()); ptr.extend(logits.argmax(1).detach().cpu().tolist())
    model.eval(); vloss=0.0; yv=[]; pv=[]
    with torch.inference_mode():
        for images,labels,_ in val_loader:
            images,labels=images.to(device,non_blocking=True),labels.to(device,non_blocking=True)
            logits=model(images); vloss+=criterion(logits,labels).item()*labels.size(0)
            yv.extend(labels.cpu().tolist()); pv.extend(logits.argmax(1).cpu().tolist())
    train_loss=total_loss/len(train_ds); val_loss=vloss/len(val_ds)
    train_f1=f1_score(ytr,ptr,average='macro',zero_division=0); val_f1=f1_score(yv,pv,average='macro',zero_division=0)
    scheduler.step(val_f1)
    history.append({'epoch':epoch,'train_loss':train_loss,'validation_loss':val_loss,
        'train_macro_f1':train_f1,'validation_macro_f1':val_f1,'learning_rate':optimizer.param_groups[0]['lr']})
    print(f'epoch={epoch:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f} train_macro_f1={train_f1:.4f} val_macro_f1={val_f1:.4f}')
    if val_f1>best_val_f1:
        best_val_f1=val_f1; best_epoch=epoch
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        atomic_save_checkpoint(checkpoint_payload(best_state,best_epoch,best_val_f1),checkpoint_path)
        print('Saved current best validation checkpoint to Drive.')
    elif epoch-best_epoch>=6:
        print('Early stopping: no validation macro-F1 improvement for six epochs.'); break
model.load_state_dict(best_state)
pd.DataFrame(history).to_csv(OUTPUT_DIR/'training_history.csv',index=False)
print('Training finished. Best epoch:',best_epoch,'| validation macro-F1:',best_val_f1)

In [ ]:
# HELD-OUT TEST EVALUATION — run after the training cell completes.
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,precision_recall_fscore_support,roc_auc_score
model.eval(); y_true=[]; y_pred=[]; y_prob=[]; test_ids=[]
with torch.inference_mode():
    for images,labels,image_ids in test_loader:
        logits=model(images.to(device,non_blocking=True)); probabilities=torch.softmax(logits,dim=1).cpu().numpy()
        y_true.extend(labels.numpy().tolist()); y_pred.extend(logits.argmax(1).cpu().tolist())
        y_prob.extend(probabilities.tolist()); test_ids.extend(list(image_ids))
y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); y_prob=np.asarray(y_prob)
cm=confusion_matrix(y_true,y_pred,labels=list(range(len(CLASS_NAMES))))
per=precision_recall_fscore_support(y_true,y_pred,labels=list(range(len(CLASS_NAMES))),zero_division=0)
macro=precision_recall_fscore_support(y_true,y_pred,average='macro',zero_division=0)
weighted=precision_recall_fscore_support(y_true,y_pred,average='weighted',zero_division=0)
metrics={'accuracy':float(accuracy_score(y_true,y_pred)),'macro_precision':float(macro[0]),
    'macro_sensitivity_recall':float(macro[1]),'macro_f1':float(macro[2]),
    'weighted_precision':float(weighted[0]),'weighted_sensitivity_recall':float(weighted[1]),'weighted_f1':float(weighted[2]),
    'per_class':{},'confusion_matrix_rows_true_columns_predicted':cm.tolist(),'test_count':int(len(y_true)),
    'split_grouping':split_basis,'patient_independent_split_verified':bool(patient_grouping_verified)}
for i,name in enumerate(CLASS_NAMES):
    tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
    metrics['per_class'][name]={'precision':float(per[0][i]),'sensitivity_recall':float(per[1][i]),
        'f1':float(per[2][i]),'specificity':float(tn/(tn+fp)) if tn+fp else None,'support':int(per[3][i])}
if set(np.unique(y_true))==set(range(len(CLASS_NAMES))):
    metrics['roc_auc_ovr_macro']=float(roc_auc_score(y_true,y_prob,multi_class='ovr',average='macro'))
else:
    metrics['roc_auc_ovr_macro']=None
    print('ROC-AUC omitted because the held-out test set lacks at least one class.')
print(json.dumps(metrics,indent=2))
print(classification_report(y_true,y_pred,labels=list(range(len(CLASS_NAMES))),target_names=CLASS_NAMES,zero_division=0))

In [ ]:
# EXPORT — outputs go to Drive so they survive Colab runtime deletion.
from datetime import datetime,timezone
import seaborn as sns

metrics['model_version']=MODEL_VERSION; metrics['selected_epoch']=int(best_epoch)
metrics['probabilities_calibrated']=False
metrics['caution']='Research evaluation only; not a clinical diagnosis. Patient-level independence is unverified when no patient ID key exists.'
metrics['trained_at_utc']=datetime.now(timezone.utc).isoformat()
# Final checkpoint uses the exact format consumed by training/src/inference.py.
atomic_save_checkpoint(checkpoint_payload({k:v.detach().cpu() for k,v in model.state_dict().items()},best_epoch,best_val_f1),checkpoint_path)
(OUTPUT_DIR/'class_mapping.json').write_text(json.dumps({str(i):name for i,name in enumerate(CLASS_NAMES)},indent=2),encoding='utf-8')
(OUTPUT_DIR/'preprocessing_config.json').write_text(json.dumps(PREPROCESSING_CONFIG,indent=2),encoding='utf-8')
(OUTPUT_DIR/'evaluation_metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
predictions=pd.DataFrame({'image_id':test_ids,'source_label':[CLASS_NAMES[i] for i in y_true],
    'model_prediction':[CLASS_NAMES[i] for i in y_pred],
    **{f'score_{name.lower()}':y_prob[:,i] for i,name in enumerate(CLASS_NAMES)}})
predictions.to_csv(OUTPUT_DIR/'test_predictions.csv',index=False)
fig,ax=plt.subplots(figsize=(6,5)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,ax=ax)
ax.set(xlabel='Predicted source class',ylabel='True source class',title='Held-out test confusion matrix')
plt.tight_layout(); fig.savefig(OUTPUT_DIR/'confusion_matrix.png',dpi=160); plt.show()
print('Saved checkpoint:',checkpoint_path)
print('Saved evaluation, split, audit, prediction and plot files:',OUTPUT_DIR)
print('No Phase 3 result is complete until this cell finishes successfully on the full archive.')

## Outputs, interpretation, and limits

Before training, audit and split files are under `/content/OsteoLensRuns/BTXRD`. After you approve training, cell 12 mounts Drive and copies the audit/split there. Training outputs include the checkpoint, class mapping, preprocessing configuration, training history, held-out test predictions and evaluation metrics, plus the confusion-matrix plot.

Softmax scores are uncalibrated research outputs, not clinical probabilities. If no complete patient/subject identifier exists, duplicate grouping cannot prove patient-independent evaluation. Annotation JSON is audited but not used as classification labels; localization and Grad-CAM remain later-phase work. The archive and extraction remain in Colab's temporary storage and must be downloaded again after runtime deletion.
